## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATAROOT = '/content/drive/MyDrive/nuscenes'
CKPT_DIR = '/content/drive/MyDrive/bev_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Verify data exists
print('nuScenes folder contents:', os.listdir(DATAROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
nuScenes folder contents: ['.v1.0-mini.txt', 'LICENSE', 'v1.0-mini', 'samples', 'maps', 'sweeps']


## Cell 2 — Install Dependencies

In [ ]:
!pip install nuscenes-devkit pyquaternion pytorch-lightning timm scipy wandb -q

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
object address  : 0x7c21534e51e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [ ]:
# Clean slate inside current runtime
!pip uninstall -y numpy scipy nuscenes-devkit >/dev/null 2>&1

# Install compatible modern stack
!pip install -q numpy==2.0.2
!pip install -q scipy==1.13.1
!pip install -q nuscenes-devkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymc 5.28.1 requires scipy>=1.4.1, which is not installed.
imbalanced-learn 0.14.1 requires scipy<2,>=1.11.4, which is not installed.
access 1.1.10.post3 requires scipy>=1.14.1, which is not installed.
matplotlib-venn 1.1.2 requires scipy, which is not installed.
mapclassify 2.10.0 requires scipy>=1.12, which is not installed.
jaxlib 0.7.2 requires scipy>=1.13, which is not installed.
shap 0.51.0 requires scipy, which is not installed.
inequality 1.1.2 requires scipy>=1.12, which is not installed.
quantecon 0.11.1 requires scipy>=1.5.0, which is not installed.
xarray-einstats 0.10.0 requires scipy>=1.13, which is not installed.
albumentations 2.0.8 requires 

In [ ]:
import numpy
import scipy
from nuscenes.nuscenes import NuScenes

print("Environment stable ✅")

ModuleNotFoundError: No module named 'numpy.strings'

In [ ]:
!pip uninstall -y numpy torchvision
!pip install -q "numpy<2.0"
!pip install -q torchvision

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: torchvision 0.17.2
Uninstalling torchvision-0.17.2:
  Successfully uninstalled torchvision-0.17.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have num

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

/bin/bash: line 1: nvidia-smi: command not found


## Cell 3 — Config

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # Data
    dataroot: str = DATAROOT
    version:  str = 'v1.0-mini'    # switch from mini

    # Image resolution
    img_h: int = 224
    img_w: int = 480

    # BEV grid [min_m, max_m, step_m]
    x_bound: List[float] = field(default_factory=lambda: [-50.0, 50.0, 0.5])
    y_bound: List[float] = field(default_factory=lambda: [-50.0, 50.0, 0.5])
    z_bound: List[float] = field(default_factory=lambda: [ -3.0,  1.0, 4.0])  # ← fixed, see note

    # LSS depth bins
    d_bound: List[float] = field(default_factory=lambda: [1.0, 49.0, 2.0])    # 24 bins

    # Training
    batch_size:   int   = 8      # bump on your better GPU, drop to 4 if OOM
    num_workers:  int   = 8      # scale to your CPU cores
    max_epochs:   int   = 50     # sufficient on full trainval
    lr:           float = 3e-4   # slightly higher, warmup will handle cold start
    weight_decay: float = 1e-4
    pos_weight:   float = 12.0

    # W&B
    wandb_project: str = 'nuscenes-bev'

    @property
    def bev_h(self): return int((self.y_bound[1] - self.y_bound[0]) / self.y_bound[2])
    @property
    def bev_w(self): return int((self.x_bound[1] - self.x_bound[0]) / self.x_bound[2])

cfg = Config()
print(f'BEV grid : {cfg.bev_h} x {cfg.bev_w} cells')
print(f'Image    : {cfg.img_h} x {cfg.img_w}')
print(f'Depth D  : {int((cfg.d_bound[1]-cfg.d_bound[0])/cfg.d_bound[2])} bins')

BEV grid : 200 x 200 cells
Image    : 224 x 480
Depth D  : 24 bins


## Cell 4 — Dataset

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.splits import create_splits_scenes
from nuscenes.utils.data_classes import LidarPointCloud
from scipy.ndimage import binary_dilation

CAMERAS = [
    'CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_FRONT_LEFT',
    'CAM_BACK',  'CAM_BACK_RIGHT',  'CAM_BACK_LEFT',
]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class NuScenesDataset(Dataset):
    def __init__(self, cfg, split):
        self.cfg    = cfg
        self.split  = split
        self.img_h  = cfg.img_h
        self.img_w  = cfg.img_w
        self.augment = (split == 'train')

        x_min, x_max, dx = cfg.x_bound
        y_min, y_max, dy = cfg.y_bound
        self.bev_w = int((x_max - x_min) / dx)
        self.bev_h = int((y_max - y_min) / dy)

        print(f'Loading nuScenes {cfg.version} [{split}]...')
        self.nusc = NuScenes(version=cfg.version, dataroot=cfg.dataroot, verbose=False)

        split_scenes = set(create_splits_scenes()[split])
        self.samples = []
        for scene in self.nusc.scene:
            if scene['name'] not in split_scenes:
                continue
            token = scene['first_sample_token']
            while token:
                s = self.nusc.get('sample', token)
                self.samples.append(s)
                token = s['next']
        print(f'  -> {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def _cam_to_ego_matrix(self, cs):
        R = Quaternion(cs['rotation']).rotation_matrix.astype(np.float32)
        t = np.array(cs['translation'], dtype=np.float32)
        T = np.eye(4, dtype=np.float32)
        T[:3, :3] = R
        T[:3, 3]  = t
        return T

    def _augment_image(self, img):
        """
        Photometric augmentation only — no flips or crops
        since those break camera-to-ego geometry.
        """
        if random.random() > 0.5:
            img = TF.adjust_brightness(img, random.uniform(0.7, 1.3))
        if random.random() > 0.5:
            img = TF.adjust_contrast(img, random.uniform(0.7, 1.3))
        if random.random() > 0.5:
            img = TF.adjust_saturation(img, random.uniform(0.7, 1.3))
        if random.random() > 0.5:
            img = TF.adjust_hue(img, random.uniform(-0.05, 0.05))
        if random.random() > 0.3:
            # random gaussian blur — simulates different camera sharpness
            ks = random.choice([3, 5])
            img = TF.gaussian_blur(img, kernel_size=ks, sigma=random.uniform(0.1, 1.5))
        return img

    def get_camera_info(self, sample):
        imgs, Ks, T_c2e = [], [], []
        for cam in CAMERAS:
            sd  = self.nusc.get('sample_data', sample['data'][cam])
            cs  = self.nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
            img = Image.open(
                os.path.join(self.nusc.dataroot, sd['filename'])
            ).convert('RGB')
            ow, oh = img.size
            img = img.resize((self.img_w, self.img_h), Image.Resampling.BILINEAR)

            if self.augment:
                img = self._augment_image(img)

            img_t = (TF.to_tensor(img) - _MEAN) / _STD
            imgs.append(img_t)

            K = np.array(cs['camera_intrinsic'], dtype=np.float32)
            K[0, :] *= self.img_w / ow
            K[1, :] *= self.img_h / oh
            Ks.append(torch.from_numpy(K))
            T_c2e.append(torch.from_numpy(self._cam_to_ego_matrix(cs)))

        return torch.stack(imgs), torch.stack(Ks), torch.stack(T_c2e)

    def get_bev_occupancy(self, sample):
        sd  = self.nusc.get('sample_data', sample['data']['LIDAR_TOP'])
        cs  = self.nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
        pc  = LidarPointCloud.from_file(
            os.path.join(self.nusc.dataroot, sd['filename'])
        )
        pc.rotate(Quaternion(cs['rotation']).rotation_matrix)
        pc.translate(np.array(cs['translation']))
        pts = pc.points[:3].T

        xb, yb, zb = self.cfg.x_bound, self.cfg.y_bound, self.cfg.z_bound
        mask = (
            (pts[:, 0] >= xb[0]) & (pts[:, 0] < xb[1]) &
            (pts[:, 1] >= yb[0]) & (pts[:, 1] < yb[1]) &
            (pts[:, 2] >= zb[0]) & (pts[:, 2] < zb[1])
        )
        pts = pts[mask]
        ix  = np.clip(((pts[:, 0] - xb[0]) / xb[2]).astype(int), 0, self.bev_w - 1)
        iy  = np.clip(((pts[:, 1] - yb[0]) / yb[2]).astype(int), 0, self.bev_h - 1)

        occ = np.zeros((self.bev_h, self.bev_w), dtype=np.float32)
        occ[iy, ix] = 1.0

        # Dilate to thicken sparse LiDAR points into learnable targets
        # only during training — val/test stay clean for accurate IoU measurement
        if self.augment:
            iters = random.choice([1, 2])
            occ = binary_dilation(occ, iterations=iters).astype(np.float32)

        return torch.from_numpy(occ).unsqueeze(0)

    def __getitem__(self, idx):
        s = self.samples[idx]
        imgs, K, T = self.get_camera_info(s)
        bev = self.get_bev_occupancy(s)
        return {'imgs': imgs, 'intrinsics': K, 'cam_to_ego': T, 'bev': bev}


print('Dataset class defined.')

Dataset class defined.


## Cell 5 — Model (LSS)

In [ ]:
import timm
import torch.nn as nn
import torch.nn.functional as F

# ── Backbone ──────────────────────────────────────────────────────────────
class ImageBackbone(nn.Module):
    def __init__(self, model_name='efficientnet_b0', out_channels=64):
        super().__init__()
        self.encoder = timm.create_model(
            model_name, pretrained=True, features_only=True,
            out_indices=(1, 2, 3)  # stride 4,8,16 instead of 8,16,32
        )
        feat_ch = sum(self.encoder.feature_info.channels())
        self.reduce = nn.Sequential(
            nn.Conv2d(feat_ch, out_channels*2, 1, bias=False),
            nn.BatchNorm2d(out_channels*2), nn.ReLU(True),
            nn.Conv2d(out_channels*2, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(True),
        )
        self.out_channels = out_channels

    def forward(self, x):
        feats = self.encoder(x)
        h, w  = feats[0].shape[2:]  # now stride-4 resolution
        feats = [F.interpolate(f, (h, w), mode='bilinear', align_corners=False) for f in feats]
        return self.reduce(torch.cat(feats, dim=1))


# ── Depth Head (ASPP for wider receptive field) ───────────────────────────
class DepthHead(nn.Module):
    def __init__(self, in_ch, D):
        super().__init__()
        mid = in_ch // 2
        # ASPP-lite: parallel dilated convs to capture context at multiple scales
        self.b0 = nn.Sequential(nn.Conv2d(in_ch, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.b1 = nn.Sequential(nn.Conv2d(in_ch, mid, 3, padding=2,  dilation=2,  bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.b2 = nn.Sequential(nn.Conv2d(in_ch, mid, 3, padding=4,  dilation=4,  bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.b3 = nn.Sequential(nn.Conv2d(in_ch, mid, 3, padding=8,  dilation=8,  bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.fuse = nn.Sequential(
            nn.Conv2d(mid*4, in_ch, 1, bias=False), nn.BatchNorm2d(in_ch), nn.ReLU(True),
            nn.Conv2d(in_ch, D, 1),
        )

    def forward(self, x):
        x = torch.cat([self.b0(x), self.b1(x), self.b2(x), self.b3(x)], dim=1)
        return self.fuse(x).softmax(dim=1)


# ── LSS View Transformer ──────────────────────────────────────────────────
class LSSViewTransformer(nn.Module):
    def __init__(self, feat_ch, bev_ch, d_bound, x_bound, y_bound, z_bound, img_h, img_w):
        super().__init__()
        self.x_bound, self.y_bound, self.z_bound = x_bound, y_bound, z_bound

        d_arr  = np.arange(d_bound[0], d_bound[1], d_bound[2], dtype=np.float32)
        self.D = len(d_arr)
        self.register_buffer('d_values', torch.from_numpy(d_arr))

        self.bev_h  = int((y_bound[1] - y_bound[0]) / y_bound[2])
        self.bev_w  = int((x_bound[1] - x_bound[0]) / x_bound[2])
        self.feat_h = img_h // 4   # stride 4 now
        self.feat_w = img_w // 4

        ys = torch.linspace(0, self.feat_h-1, self.feat_h)
        xs = torch.linspace(0, self.feat_w-1, self.feat_w)
        yy, xx = torch.meshgrid(ys, xs, indexing='ij')
        pg = torch.stack([xx, yy, torch.ones_like(xx)], -1)  # (H, W, 3)
        self.register_buffer('pixel_grid', pg)

        self.depth_head = DepthHead(feat_ch, self.D)
        self.bev_agg = nn.Sequential(
            nn.Conv2d(feat_ch, bev_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(bev_ch), nn.ReLU(True),
        )

    def get_geometry(self, intrinsics, cam_to_ego):
        B, N    = intrinsics.shape[:2]
        D, H, W = self.D, self.feat_h, self.feat_w
        K_inv   = torch.inverse(intrinsics)                             # (B,N,3,3)

        pg_flat = self.pixel_grid.reshape(H*W, 3).T                    # (3,H*W)
        pg_flat = pg_flat.unsqueeze(0).expand(B*N, 3, H*W)
        rays    = torch.bmm(K_inv.reshape(B*N,3,3), pg_flat)           # (B*N,3,H*W)
        rays    = rays.permute(0,2,1).reshape(B,N,H,W,3).unsqueeze(2)  # (B,N,1,H,W,3)

        d      = self.d_values.view(1,1,D,1,1,1)
        p_cam  = rays * d                                               # (B,N,D,H,W,3)

        ones   = torch.ones(*p_cam.shape[:-1], 1, device=intrinsics.device)
        p_h    = torch.cat([p_cam, ones], -1)                          # (B,N,D,H,W,4)
        p_flat = p_h.reshape(B*N, D*H*W, 4).permute(0,2,1)
        p_ego  = torch.bmm(cam_to_ego.reshape(B*N,4,4), p_flat)
        return  p_ego[:,:3].permute(0,2,1).reshape(B,N,D,H,W,3)

    def voxel_pooling(self, p_ego, features, depth_probs):
        B, N, D, H, W, _ = p_ego.shape
        C = features.shape[-1]
        x_min, _, dx = self.x_bound
        y_min, _, dy = self.y_bound

        ix    = ((p_ego[..., 0] - x_min) / dx).long()
        iy    = ((p_ego[..., 1] - y_min) / dy).long()
        valid = (ix >= 0) & (ix < self.bev_w) & (iy >= 0) & (iy < self.bev_h)

        feat_exp = features.unsqueeze(2).expand(B, N, D, H, W, C)
        weighted = (feat_exp * depth_probs.unsqueeze(-1)).contiguous()  # (B,N,D,H,W,C)

        bev    = torch.zeros(B, C, self.bev_h, self.bev_w, device=features.device)
        b_idx  = torch.arange(B, device=features.device).view(B,1,1,1,1).expand(B,N,D,H,W)
        flat   = (b_idx * self.bev_h * self.bev_w + iy * self.bev_w + ix)  # (B,N,D,H,W)

        v      = valid.reshape(-1)
        flat_v = flat.reshape(-1)[v]
        fw     = weighted.reshape(-1, C)[v]                             # (M, C)

        # fully vectorized scatter — no python loop
        bev.view(B * self.bev_h * self.bev_w, C).scatter_add_(
            0,
            flat_v.unsqueeze(1).expand(-1, C),
            fw
        )
        return bev.view(B, C, self.bev_h, self.bev_w)

    def forward(self, features, intrinsics, cam_to_ego):
        B, N, C, H, W = features.shape
        p_ego       = self.get_geometry(intrinsics, cam_to_ego)
        depth_probs = self.depth_head(features.view(B*N, C, H, W)).view(B, N, self.D, H, W)
        feats_hwc   = features.permute(0, 1, 3, 4, 2)
        bev         = self.voxel_pooling(p_ego, feats_hwc, depth_probs)
        return self.bev_agg(bev)


# ── BEV Decoder (deeper + residual) ──────────────────────────────────────
class BEVDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch,    in_ch*2, 3, padding=1, bias=False), nn.BatchNorm2d(in_ch*2), nn.ReLU(True),
            nn.Conv2d(in_ch*2, in_ch*2, 3, padding=1, bias=False), nn.BatchNorm2d(in_ch*2), nn.ReLU(True),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(in_ch*2, in_ch, 3, padding=1, bias=False), nn.BatchNorm2d(in_ch), nn.ReLU(True),
            nn.Conv2d(in_ch,   in_ch, 3, padding=1, bias=False), nn.BatchNorm2d(in_ch), nn.ReLU(True),
        )
        # residual projection from input to block2 output dimension
        self.skip = nn.Conv2d(in_ch*2, in_ch, 1, bias=False)
        self.head  = nn.Conv2d(in_ch, 1, 1)

    def forward(self, x):
        x  = self.block1(x)
        x  = self.block2(x) + self.skip(x)  # residual
        return self.head(x)


# ── Full Model ────────────────────────────────────────────────────────────
class BEVModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        C = 64
        self.backbone         = ImageBackbone('efficientnet_b0', out_channels=C)
        self.view_transformer = LSSViewTransformer(
            feat_ch=C, bev_ch=C,
            d_bound=cfg.d_bound, x_bound=cfg.x_bound,
            y_bound=cfg.y_bound, z_bound=cfg.z_bound,
            img_h=cfg.img_h,    img_w=cfg.img_w,
        )
        self.decoder = BEVDecoder(C)

    def forward(self, imgs, intrinsics, cam_to_ego):
        B, N, C, H, W = imgs.shape
        feats = self.backbone(imgs.view(B*N, C, H, W))
        _, fC, fH, fW = feats.shape
        feats = feats.view(B, N, fC, fH, fW)
        bev   = self.view_transformer(feats, intrinsics, cam_to_ego)
        return self.decoder(bev)


print('Model classes defined.')

Model classes defined.


## Cell 6 — Loss

In [ ]:
class FocalDistanceLoss(nn.Module):
    def __init__(self, cfg, alpha=0.25, gamma=2.0):
        super().__init__()
        x_min, x_max, dx = cfg.x_bound
        y_min, y_max, dy = cfg.y_bound
        xs = torch.linspace(x_min+dx/2, x_max-dx/2, cfg.bev_w)
        ys = torch.linspace(y_min+dy/2, y_max-dy/2, cfg.bev_h)
        yy, xx = torch.meshgrid(ys, xs, indexing='ij')
        dist = (xx**2 + yy**2).sqrt().clamp(min=1.0)
        w = 1.0 / dist
        w = w / w.mean()
        self.register_buffer('w', w.unsqueeze(0).unsqueeze(0))
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = cfg.pos_weight

    def forward(self, logits, target):
        pw  = torch.tensor(self.pos_weight, device=logits.device, dtype=logits.dtype)
        bce = F.binary_cross_entropy_with_logits(
            logits, target, pos_weight=pw, reduction='none'
        )
        prob   = logits.sigmoid()
        pt     = torch.where(target == 1, prob, 1 - prob)
        alpha  = torch.where(target == 1,
                     torch.tensor(self.alpha, device=logits.device),
                     torch.tensor(1 - self.alpha, device=logits.device))
        focal  = alpha * (1 - pt) ** self.gamma * bce
        return (focal * self.w).mean()

print('Loss defined.')

Loss defined.


## Cell 7 — Lightning Module

In [ ]:
import pytorch_lightning as pl
import wandb
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

class BEVLightningModule(pl.LightningModule):
    def __init__(self, cfg):
        super().__init__()
        self.cfg     = cfg
        self.model   = BEVModel(cfg)
        self.loss_fn = FocalDistanceLoss(cfg)
        self.save_hyperparameters()

    def forward(self, batch):
        return self.model(batch['imgs'], batch['intrinsics'], batch['cam_to_ego'])

    def _step(self, batch, stage):
        logits = self(batch)
        target = batch['bev']
        loss   = self.loss_fn(logits, target)
        with torch.no_grad():
            pred   = (logits.sigmoid() > 0.5).float()
            tp     = (pred * target).sum()
            fp     = (pred * (1-target)).sum()
            fn     = ((1-pred) * target).sum()
            iou    = tp / (tp + fp + fn + 1e-6)
            recall = tp / (tp + fn + 1e-6)
        self.log(f'{stage}/loss',   loss,   prog_bar=True)
        self.log(f'{stage}/iou',    iou,    prog_bar=True)
        self.log(f'{stage}/recall', recall)
        return loss, logits, target

    def training_step(self, batch, batch_idx):
        loss, _, _ = self._step(batch, 'train')
        return loss

    def validation_step(self, batch, batch_idx):
        loss, logits, target = self._step(batch, 'val')
        if batch_idx == 0 and isinstance(self.logger, WandbLogger):
            self._log_bev(logits, target)

    def on_train_epoch_start(self):
      if self.current_epoch < 5:
          for p in self.model.backbone.parameters():
              p.requires_grad = False
      else:
          for p in self.model.backbone.parameters():
              p.requires_grad = True

    def _log_bev(self, logits, target, n=2):
        probs = logits[:n].sigmoid().squeeze(1).cpu().numpy()
        gts   = target[:n].squeeze(1).cpu().numpy()
        imgs  = []
        for i in range(probs.shape[0]):
            panel = np.concatenate([probs[i], gts[i]], axis=1)   # pred | gt
            imgs.append(wandb.Image(panel, caption=f'[{i}] pred | gt'))
        self.logger.experiment.log({'val/bev': imgs, 'epoch': self.current_epoch})

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.cfg.lr,
            weight_decay=self.cfg.weight_decay
        )

        # Warmup for first 500 steps, then cosine decay
        warmup = LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=500)
        cosine = CosineAnnealingLR(opt, T_max=self.cfg.max_epochs, eta_min=self.cfg.lr * 0.001)
        scheduler = SequentialLR(opt, schedulers=[warmup, cosine], milestones=[500])

        return {
            'optimizer': opt,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'step',    # ← step-level for warmup to work correctly
            }
        }

print('Lightning module defined.')

Lightning module defined.


## Cell 8 — Quick Sanity Check (no training)
Run this first to confirm shapes are correct before launching full training.

In [ ]:
print('--- Sanity check ---')
ds_check = NuScenesDataset(cfg, split='train')
sample   = ds_check[0]
print('imgs      :', sample['imgs'].shape)        # (6, 3, 224, 480)
print('intrinsics:', sample['intrinsics'].shape)  # (6, 3, 3)
print('cam_to_ego:', sample['cam_to_ego'].shape)  # (6, 4, 4)
print('bev       :', sample['bev'].shape)         # (1, 200, 200)
print('bev occ % :', sample['bev'].mean().item() * 100, '%')

# Model forward pass
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {device}')

model = BEVModel(cfg).to(device)
batch = {k: v.unsqueeze(0).to(device) for k, v in sample.items()}

with torch.no_grad():
    out = model(batch['imgs'], batch['intrinsics'], batch['cam_to_ego'])
print('Output logits:', out.shape)   # (1, 1, 200, 200)
print('\n✅ Sanity check passed — ready to train!')

--- Sanity check ---
Loading nuScenes v1.0-mini [train]...
  -> 242 samples
imgs      : torch.Size([6, 3, 224, 480])
intrinsics: torch.Size([6, 3, 3])
cam_to_ego: torch.Size([6, 4, 4])
bev       : torch.Size([1, 200, 200])
bev occ % : 20.389999449253082 %

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Output logits: torch.Size([1, 1, 200, 200])

✅ Sanity check passed — ready to train!


In [ ]:
import torch
import pickle

path = '/content/drive/MyDrive/BEV Model'

torch.save(model.state_dict(), f'{path}/bev_model_init.pth')

with open(f'{path}/bev_cfg.pkl', 'wb') as f:
    pickle.dump(cfg, f)

print('Done.')

Done.


## Cell 9 — W&B Login
Skip with `use_wandb = False` if you just want console logs.

In [ ]:
use_wandb = True   # set False to skip

if use_wandb:
    wandb.login()   # prompts for API key — get it from wandb.ai/authorize
    print('W&B ready.')
else:
    print('Skipping W&B.')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vibhudev (vibhudev-dayananda-sagar-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready.


## Cell 10 — Train

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

pl.seed_everything(42, workers=True)

# Datasets
train_ds = NuScenesDataset(cfg, split='train')
val_ds   = NuScenesDataset(cfg, split='val')

train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=cfg.num_workers, pin_memory=True,
                      drop_last=True, persistent_workers=True)   # ← persistent_workers
val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                      num_workers=cfg.num_workers, pin_memory=True,
                      persistent_workers=True)

# Module
module = BEVLightningModule(cfg)

# Logger
logger = WandbLogger(project=cfg.wandb_project, save_dir='/content/') if use_wandb else False

# Callbacks
callbacks = [
    ModelCheckpoint(
        monitor='val/iou', mode='max', save_top_k=3, save_last=True,
        dirpath=CKPT_DIR,
        filename='bev-{epoch:02d}-iou{val/iou:.4f}',
    ),
    LearningRateMonitor(logging_interval='step'),   # ← step not epoch, warmup is step-level
]

trainer = pl.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator='gpu', devices=1,
    precision='16-mixed',
    logger=logger,
    callbacks=callbacks,
    log_every_n_steps=10,          # ← 5 is too frequent on large dataset, chattier logs
    val_check_interval=1.0,        # ← once per epoch on full dataset, 0.5 is overkill
    gradient_clip_val=1.0,
)

trainer.fit(module, train_dl, val_dl)

INFO:lightning_fabric.utilities.seed:Seed set to 42


Loading nuScenes v1.0-mini [train]...
  -> 242 samples
Loading nuScenes v1.0-mini [val]...
  -> 162 samples


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platf

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/bev_checkpoints exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ BEVModel          │  4.1 M │ train │     0 │
│ 1 │ loss_fn │ FocalDistanceLoss │      0 │ train │     0 │
└───┴─────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 4.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.1 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 383                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will 
create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller 
than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader 
running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


## Cell 11 — Resume After Session Crash
Run this cell instead of Cell 10 if the session restarted.

In [ ]:
# Find the latest checkpoint
import glob
ckpts = sorted(glob.glob(f'{CKPT_DIR}/*.ckpt'))
print('Available checkpoints:')
for c in ckpts: print(' ', c)

RESUME_CKPT = f'{CKPT_DIR}/last.ckpt'   # or paste a specific path
print(f'\nWill resume from: {RESUME_CKPT}')

Available checkpoints:
  /content/drive/MyDrive/bev_checkpoints/last-v1.ckpt
  /content/drive/MyDrive/bev_checkpoints/last.ckpt

Will resume from: /content/drive/MyDrive/bev_checkpoints/last.ckpt


In [ ]:
# Rerun this only if resuming — re-run Cells 1-9 first after a session restart
trainer.fit(module, train_dl, val_dl, ckpt_path=RESUME_CKPT)

## Cell 12 — Visualise BEV Predictions

In [ ]:
import matplotlib.pyplot as plt

module.eval().to('cuda')

sample = val_ds[0]
batch  = {k: v.unsqueeze(0).cuda() for k, v in sample.items()}

with torch.no_grad():
    logits = module(batch)
    pred   = logits.sigmoid().squeeze().cpu().numpy()
    gt     = sample['bev'].squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(pred, cmap='hot', origin='lower', vmin=0, vmax=1)
axes[0].set_title('Predicted BEV Occupancy')
axes[0].set_xlabel('x (right)')  ;  axes[0].set_ylabel('y (forward)')

axes[1].imshow(gt, cmap='hot', origin='lower', vmin=0, vmax=1)
axes[1].set_title('Ground Truth BEV (LiDAR)')
axes[1].set_xlabel('x (right)')  ;  axes[1].set_ylabel('y (forward)')

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/bev_sample1.png', dpi=150)
plt.show()
print('Saved to Drive.')

NameError: name 'module' is not defined

In [ ]:
print(pred.min(), pred.max())

0.0001316673 0.9999608


In [ ]:
# Quick IoU evaluation on val set
# Add this after training instead of fixed 0.5
best_iou, best_thresh = 0, 0.5
for thresh in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    total_tp = total_fp = total_fn = 0
    with torch.no_grad():
        for batch in val_dl:
            batch  = {k: v.cuda() for k, v in batch.items()}
            logits = module(batch)
            pred   = (logits.sigmoid() > thresh).float()
            target = batch['bev']
            total_tp += (pred * target).sum().item()
            total_fp += (pred * (1 - target)).sum().item()
            total_fn += ((1 - pred) * target).sum().item()
    iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
    print(f'thresh={thresh:.2f} → IoU={iou*100:.2f}%')
    if iou > best_iou:
        best_iou, best_thresh = iou, thresh
print(f'\nBest: thresh={best_thresh} → IoU={best_iou*100:.2f}%')

with torch.no_grad():
    for batch in val_dl:
        batch = {k: v.cuda() for k, v in batch.items()}
        logits = module(batch)
        pred   = (logits.sigmoid() > 0.5).float()
        target = batch['bev']

        total_tp += (pred * target).sum().item()
        total_fp += (pred * (1 - target)).sum().item()
        total_fn += ((1 - pred) * target).sum().item()

iou = total_tp / (total_tp + total_fp + total_fn + 1e-6)
print(f'Validation IoU: {iou:.4f} ({iou*100:.2f}%)')

In [ ]:
import pickle

with open('/content/drive/MyDrive/BEV Model/bev_cfg.pkl', 'wb') as f:
    pickle.dump(cfg, f)

In [ ]:
torch.save(module.model.state_dict(), '/content/drive/MyDrive/BEV Model/bev_model_final3.pth')